# Hybrid Pipeline — Quantum Ledger (Original)

**Objective key:** `hybrid` &nbsp;·&nbsp; **Family:** Hybrid &nbsp;·&nbsp; **Speed:** Compute-intensive

## Algorithm

Original **Quantum Ledger 3-stage pipeline** (runs entirely on CPU; created for this project):

| Stage | Method | Role |
|-------|--------|------|
| 1 — IC Screening | Classical | Rank all N assets by Information Coefficient (μ/σ); keep top `K_screen` |
| 2 — QUBO Selection | Quantum-inspired (SA) | Binary asset selection via simulated annealing; output `K_select` assets |
| 3 — Markowitz Allocation | Classical | Convex Max-Sharpe QP on the `K_select` chosen assets |

**Stage 2 uses QUBO + simulated annealing** (same formulation as `qubo_sa` objective).  
IBM Quantum **is not** used in the current pipeline; QAOA on Runtime is a planned future upgrade for stage 2.

## Papers

- **Foundational (stage 3):** Markowitz (1952) — *Portfolio Selection* — Journal of Finance  
  https://doi.org/10.1111/j.1540-6261.1952.tb01525.x
- **Foundational (stage 2 basis):** Orús et al. (2019) — *Quantum Computing for Finance*  
  https://arxiv.org/abs/1811.03975 &nbsp;·&nbsp; **PDF:** https://arxiv.org/pdf/1811.03975
- **Modern (related reading):** Cerezo et al. (2021) — *Variational Quantum Algorithms* — Nature Reviews Physics  
  https://arxiv.org/abs/2012.09265 &nbsp;·&nbsp; **PDF:** https://arxiv.org/pdf/2012.09265  
  *(Survey of hybrid QC landscape. This repo's 3-stage pipeline is an implementation synthesis.)*

## Assumptions

- `mu` and `Sigma` are annualised.
- `K_screen=None` (auto ≈ 60% of N), `K_select=None` (auto ≈ 50% of K_screen).
- `lambda_risk=1.0`, `gamma=8.0`, `weight_min=0.005`, `weight_max=0.30`, `seed=42`.
- `stage_info` contains diagnostic fields from each stage (screened count, QUBO objective, Sharpe).

In [ ]:
import sys
from pathlib import Path

def _find_root() -> Path:
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (p / "api" / "app.py").is_file():
            return p
    raise RuntimeError("Cannot find repo root")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Repo root: {ROOT}")

In [ ]:
import numpy as np
from services.portfolio_optimizer import run_optimization

rng = np.random.default_rng(42)
n = 15  # full universe; stage 1 will screen down, stage 2 will select further
mu = rng.uniform(0.06, 0.22, n)
cov_raw = rng.standard_normal((n, n))
Sigma = cov_raw @ cov_raw.T / n + np.eye(n) * 0.04
asset_names = [f"ASSET_{i+1:02d}" for i in range(n)]
print(f"Universe: {n} assets")
print("mu range:", mu.min().round(4), "—", mu.max().round(4))

In [ ]:
result = run_optimization(
    returns=mu,
    covariance=Sigma,
    objective="hybrid",
    asset_names=asset_names,
    K_screen=9,   # stage 1: keep 9 out of 15
    K_select=5,   # stage 2: QUBO picks 5 out of 9
    lambda_risk=1.0,
    gamma=8.0,
    weight_min=0.005,
    weight_max=0.30,
    seed=42,
)

print(f"\nObjective:       {result.objective}")
print(f"Expected return: {result.expected_return:.4f}")
print(f"Volatility:      {result.volatility:.4f}")
print(f"Sharpe ratio:    {result.sharpe_ratio:.4f}")
print("\nFinal portfolio weights (stage 3 Markowitz):")
active = [(name, w) for name, w in zip(asset_names, result.weights) if w > 1e-6]
for name, w in active:
    print(f"  {name}: {w:.4f}")

In [ ]:
# Stage diagnostics (if available)
if hasattr(result, 'stage_info') and result.stage_info:
    info = result.stage_info
    print("\nStage diagnostics:")
    print(f"  Stage 1 screened count:  {info.get('stage1_screened_count', 'n/a')}")
    print(f"  Stage 2 selected assets: {info.get('stage2_selected_names', 'n/a')}")
    print(f"  Stage 2 QUBO objective:  {info.get('stage2_qubo_obj', 'n/a')}")
    print(f"  Stage 3 Sharpe:          {info.get('stage3_sharpe', 'n/a')}")
else:
    print("\nStage info not available on this result object.")

In [ ]:
# Compare pipeline stages against classical baselines
mkt = run_optimization(returns=mu, covariance=Sigma, objective="markowitz",
                       weight_min=0.005, weight_max=0.30, seed=42)
ew  = run_optimization(returns=mu, covariance=Sigma, objective="equal_weight")

print("Sharpe comparison:")
print(f"  Hybrid pipeline: {result.sharpe_ratio:.4f}")
print(f"  Markowitz:       {mkt.sharpe_ratio:.4f}")
print(f"  Equal Weight:    {ew.sharpe_ratio:.4f}")
print(f"\nHeld assets: Hybrid={len(active)}  |  Markowitz={sum(1 for w in mkt.weights if w>1e-6)}  |  EW={n}")